# Machine Learning-Based Network Intrusion Detection System
**Project:** IBM SkillsBuild Cyber Security 2026 — BharatCares  
**Student Name:** Sureka MJ  
**Register Number:** 2503617814922055  
**College:** Sona College of Technology  
**Department:** B.E. CSE (Cybersecurity)  
**GitHub:** https://github.com/Sureka-MJ  
**Dataset:** NSL-KDD Network Intrusion Detection Dataset  
**Goal:** Detect whether a network connection is **normal** or an **attack** using machine learning.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, RocCurveDisplay

import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded successfully.')

## 2. Load Dataset
Download `KDDTrain+.txt` and `KDDTest+.txt` from the NSL-KDD Kaggle dataset and keep them in the same folder as this notebook.
Dataset link: https://www.kaggle.com/datasets/hassan06/nslkdd

In [ ]:
columns = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
    'wrong_fragment','urgent','hot','num_failed_logins','logged_in','num_compromised',
    'root_shell','su_attempted','num_root','num_file_creations','num_shells',
    'num_access_files','num_outbound_cmds','is_host_login','is_guest_login','count',
    'srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate',
    'same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count',
    'dst_host_srv_count','dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate','dst_host_serror_rate',
    'dst_host_srv_serror_rate','dst_host_rerror_rate','dst_host_srv_rerror_rate',
    'label','difficulty'
]

train_df = pd.read_csv('KDDTrain+.txt', names=columns)
test_df = pd.read_csv('KDDTest+.txt', names=columns)

print('Training data shape:', train_df.shape)
print('Testing data shape:', test_df.shape)
train_df.head()

## 3. Data Understanding and Cleaning

In [ ]:
print(train_df.info())
print('\nMissing values:', train_df.isnull().sum().sum())
print('Duplicate rows:', train_df.duplicated().sum())
print('\nClass distribution:')
print(train_df['label'].value_counts().head())

In [ ]:
train_df['target'] = np.where(train_df['label'] == 'normal', 0, 1)
test_df['target'] = np.where(test_df['label'] == 'normal', 0, 1)

train_df = train_df.drop(columns=['label', 'difficulty'])
test_df = test_df.drop(columns=['label', 'difficulty'])

plt.figure(figsize=(5, 4))
sns.countplot(x='target', data=train_df)
plt.title('Normal vs Attack Connections')
plt.xticks([0, 1], ['Normal', 'Attack'])
plt.show()

## 4. Feature Preparation

In [ ]:
X_train = train_df.drop(columns=['target'])
y_train = train_df['target']
X_test = test_df.drop(columns=['target'])
y_test = test_df['target']

categorical_features = ['protocol_type', 'service', 'flag']
numeric_features = [col for col in X_train.columns if col not in categorical_features]

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

print('Numeric features:', len(numeric_features))
print('Categorical features:', categorical_features)

## 5. Train Machine Learning Models

In [ ]:
logistic_model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

random_forest_model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

logistic_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)
print('Models trained successfully.')

## 6. Model Evaluation

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    results = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred)
    }
    print(f'\n===== {model_name} =====')
    print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack']))
    return results, y_pred

logistic_results, logistic_pred = evaluate_model(logistic_model, X_test, y_test, 'Logistic Regression')
rf_results, rf_pred = evaluate_model(random_forest_model, X_test, y_test, 'Random Forest')

pd.DataFrame([logistic_results, rf_results])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(confusion_matrix(y_test, logistic_pred), annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Logistic Regression Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(confusion_matrix(y_test, rf_pred), annot=True, fmt='d', cmap='Greens', ax=axes[1])
axes[1].set_title('Random Forest Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
RocCurveDisplay.from_estimator(logistic_model, X_test, y_test, name='Logistic Regression')
RocCurveDisplay.from_estimator(random_forest_model, X_test, y_test, name='Random Forest')
plt.title('ROC Curve Comparison')
plt.show()

## 7. Conclusion
The Random Forest model generally performs better than Logistic Regression for detecting network attacks because it captures complex, non-linear relationships among network-traffic features. Recall is especially important because missed attacks can create security risks.

**Prepared by:** Sureka MJ  
**Register Number:** 2503617814922055  
**Sona College of Technology**  
**B.E. CSE (Cybersecurity)**  
**GitHub:** https://github.com/Sureka-MJ